# Backup and Recovery
## Strategy 3 – Iceberg rewrite_table_path Procedure

Note this will require EMR 7.11+ and the EMR serverless application attached to subnets with internet connectivity (e.g., through a NAT)

### Configure Spark Session
Change `eu-west-1` and `us-east-1` with your main region and backup region

In [ ]:
%%configure -f
{   
    "numExecutors": 1, 
    "executorMemory": "14G",
    "executorCores": 4, 
    "conf":{
         "spark.sql.extensions":"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
         "spark.hadoop.hive.metastore.client.factory.class":"com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory",
         "spark.sql.catalog.iceberg_catalog_region1":"org.apache.iceberg.spark.SparkCatalog",
         "spark.sql.catalog.iceberg_catalog_region1.type": "glue",
         "spark.sql.catalog.iceberg_catalog_region1.glue.region":"eu-west-1",
         "spark.sql.catalog.iceberg_catalog_region1.s3.region":"eu-west-1",
         "spark.sql.catalog.iceberg_catalog_region1.client.region":"eu-west-1",
         "spark.sql.catalog.iceberg_catalog_region2":"org.apache.iceberg.spark.SparkCatalog",
         "spark.sql.catalog.iceberg_catalog_region2.type": "glue",
         "spark.sql.catalog.iceberg_catalog_region2.glue.region":"us-east-1",
         "spark.sql.catalog.iceberg_catalog_region2.s3.region":"us-east-1",
         "spark.sql.catalog.iceberg_catalog_region2.client.region":"us-east-1",
         "spark.dynamicAllocation.enabled": "false"
    }
}

### Configure variables

In [ ]:
db_name="mydb"
table_name="test_rewrite_table"
bucket_name="mybucket1"
target_bucket_name="mybucket2"
s3_prefix_source="s3://mybucket1/"
s3_prefix_target="s3://mybucket2/"
region1="eu-west-1"
region2="us-east-1"

### Utility functions

In [ ]:
# Utility function to print a tree view of an S3 prefix
def create_s3_tree(bucket_name, prefix):
    import boto3
    from collections import defaultdict

    s3 = boto3.client('s3')
    prefix = prefix.rstrip('/') + '/'
    
    # Get all objects
    paginator = s3.get_paginator('list_objects_v2')
    tree = defaultdict(dict)
    
    # Build tree structure
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                path = obj['Key'][len(prefix):] if obj['Key'].startswith(prefix) else obj['Key']
                if path:
                    parts = path.split('/')
                    current = tree
                    for part in parts:
                        if part:
                            if part not in current:
                                current[part] = {}
                            current = current[part]
    
    # Print tree with ASCII characters
    def print_tree(d, indent=0, prefix=''):
        items = sorted(d.items())
        for i, (name, subtree) in enumerate(items):
            is_last = i == len(items) - 1
            print(f"{prefix}{'`-- ' if is_last else '|-- '}{name}")
            if subtree:
                new_prefix = prefix + ('    ' if is_last else '|   ')
                print_tree(subtree, indent + 1, new_prefix)

    print(f"\n{bucket_name}/{prefix}")
    print_tree(tree)

In [ ]:
import boto3
from urllib.parse import urlparse

# Utility function to copy S3 files from a source location to a destination location
def copy_files_in_partition(partition):
    s3_client = boto3.client('s3')
    
    for row in partition:
        source_path = row["source"]
        dest_path = row["dest"]
        
        # Parse S3 URLs
        src_parsed = urlparse(source_path)
        src_bucket = src_parsed.netloc
        src_key = src_parsed.path.lstrip('/')

        dst_parsed = urlparse(dest_path)
        dst_bucket = dst_parsed.netloc
        dst_key = dst_parsed.path.lstrip('/')

        print(f"Copying {source_path} to {dest_path}")

        # Copy the object
        copy_source = {'Bucket': src_bucket, 'Key': src_key}
        s3_client.copy_object(CopySource=copy_source, Bucket=dst_bucket, Key=dst_key)
        print(f"Copied {source_path} to {dest_path}")

In [ ]:
import boto3
import os
import re
from botocore.exceptions import ClientError
# Utility functions to replicate Glue table metadata substituting the root s3 location

def get_glue_client(region_name):
    return boto3.client('glue', region_name=region_name)

def copy_table_metadata(source_database, source_table, source_region, target_region, source_bucket, target_bucket):
    glue_client = get_glue_client(source_region)
    target_glue_client = get_glue_client(target_region)
    try:
        # Get source table metadata
        source = glue_client.get_table(DatabaseName=source_database, Name=source_table)
        table_input = source['Table']

        # Prepare new table input with required fields
        new_table_input = {
            'Name': table_input['Name'],
            'StorageDescriptor': table_input['StorageDescriptor'],
            'TableType': table_input['TableType'],
            'Parameters': table_input['Parameters']
        }

        # Handle optional fields
        optional_fields = ['Owner', 'Description', 'Retention', 'ViewOriginalText',
                           'ViewExpandedText', 'TargetTable']

        for field in optional_fields:
            if field in table_input:
                new_table_input[field] = table_input[field]

        print("new_table_input BEFORE replace")
        print(new_table_input)

        # Replace bucket name in all relevant fields
        # 1. Replace in StorageDescriptor.Location
        if 'Location' in new_table_input['StorageDescriptor']:
            new_table_input['StorageDescriptor']['Location'] = new_table_input['StorageDescriptor']['Location'].replace(
                f's3://{source_bucket}/', f's3://{target_bucket}/')
        # 2. Replace in AdditionalLocations if present
        if 'AdditionalLocations' in new_table_input['StorageDescriptor']:
            new_locations = []
            for location in new_table_input['StorageDescriptor']['AdditionalLocations']:
                new_locations.append(location.replace(f's3://{source_bucket}/', f's3://{target_bucket}/'))

            new_table_input['StorageDescriptor']['AdditionalLocations'] = new_locations

        # 3. Replace in Parameters (especially important for Iceberg tables)
        if 'Parameters' in new_table_input:
            for key, value in new_table_input['Parameters'].items():
                if isinstance(value, str) and source_bucket in value:
                    new_table_input['Parameters'][key] = value.replace(
                        f's3://{source_bucket}/', f's3://{target_bucket}/')

        print("new_table_input AFTER bucket replacement:")
        print(new_table_input)

        # Check if the table already exists in the target region
        table_exists = False

        try:
            target_glue_client.get_table(DatabaseName=source_database, Name=source_table)
            table_exists = True
        except ClientError as e:
            # If the error is EntityNotFoundException, the table doesn't exist
            if e.response['Error']['Code'] == 'EntityNotFoundException':
                table_exists = False
            else:
                # For any other error, raise it
                raise e

        # Create or update the table based on whether it exists
        if table_exists:
            print(f"Table {source_database}.{source_table} already exists in target region. Updating...")
            target_glue_client.update_table(
                DatabaseName=source_database,
                TableInput=new_table_input
            )
            print(f"Table metadata successfully updated for {source_database}.{source_table}")
        else:
            print(f"Creating new table {source_database}.{source_table} in target region...")
            target_glue_client.create_table(
                DatabaseName=source_database,
                TableInput=new_table_input
            )
            print(f"Table metadata successfully created for {source_database}.{source_table}")
        return True

    except Exception as e:
        print(f"Error copying table metadata: {str(e)}")
        return False



def replicate_table_metadata(source_database, source_table, source_region, target_region, source_bucket, target_bucket):
    print(f"Starting replication of metadata of {source_database}.{source_table} from {source_region} to {target_region}")
    print(f"Replacing bucket {source_bucket} with {target_bucket}")

    if copy_table_metadata(source_database, source_table, source_region, target_region, source_bucket, target_bucket):
        print(f"Copy of {source_database}.{source_table} from {source_region} to {target_region} completed successfully")
        return True
    else:
        print(f"Copy failed: Unable to copy table metadata")
        return False

### Create db in source and target region

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS iceberg_catalog_region1.{db_name}
LOCATION 's3://{bucket_name}/{db_name}'
""")

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS iceberg_catalog_region2.{db_name}
LOCATION 's3://{target_bucket_name}/{db_name}'
""")

### Create and populate main table

In [ ]:
spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS iceberg_catalog_region1.{db_name}.{table_name}(
    id INT,
    col1 INT,
    col2 INT,
    col3 INT,
    col4 INT,
    country string
)
USING ICEBERG
PARTITIONED BY (country)
LOCATION 's3://{bucket_name}/{db_name}/{table_name}/'
""")

In [ ]:
spark.sql(f"""
    INSERT INTO iceberg_catalog_region1.{db_name}.{table_name}
    VALUES 
    (1, 100, 200, 300, 400, 'italy'),
    (2, 101, 201, 301, 401, 'france')
""")

In [ ]:
spark.sql(f"""
    INSERT INTO iceberg_catalog_region1.{db_name}.{table_name}
    VALUES 
    (3, 102, 202, 302, 402, 'italy'),
    (4, 103, 203, 303, 403, 'france')
""")

Verify table content

In [ ]:
spark.sql(f"""
SELECT * 
FROM iceberg_catalog_region1.{db_name}.{table_name} ORDER BY id
""").show(truncate=False)

This the original files structure

In [ ]:
create_s3_tree(bucket_name, f'{db_name}/{table_name}')

### Step1: Full Table Rewrite
#### A. Invoke the rewrite_table_path procedure (Full rewrite mode)

Invoke the `rewrite_table_path` procedure to create Iceberg metadata files with source_prefix substituted by target_prefix

In [ ]:
rewrite_table_path_df=spark.sql(f"""
CALL iceberg_catalog_region1.system.rewrite_table_path(
    table => '{db_name}.{table_name}', 
    source_prefix => '{s3_prefix_source}',
    target_prefix => '{s3_prefix_target}'
)
"""
)

rows = rewrite_table_path_df.collect()
file_list_location = rows[0]["file_list_location"]
latest_version = rows[0]["latest_version"]

The procedure returns a dataframe with one record and two columns:
- `latest_version`: the latest JSON metadata file rewritten by the procedure. This is useful as could be used at the next invocation of the procure in incremental mode as start_version
- `file_list_location`: the S3 path to a location that was populated by the procedure with CSV files containing the mapping of source to target paths. 

In [ ]:
rewrite_table_path_df.show(truncate=False)

The procedure creates a new copy-table-staging directory under the metadata folder containing:
1. Rewritten metadata files with paths updated from s3://mybucket1 to s3://mybucket2
2. A file-list folder with CSV files mapping source locations to destination locations. This folder is pointed by the file_list_location column mentioned above, part of the dataframe returned by the procedure.

In [ ]:
create_s3_tree(bucket_name, f'{db_name}/{table_name}')

Here an example of CSV files content under the `file_list_location` path. For each line the first column contain the source file location and the second column the dest file location (e.g., the location where the file should be copied)

In [ ]:
df=spark.read.csv(file_list_location)
df.show(truncate=False)

#### B. Copy table files in secondary region

Copy the files in the destination bucket

In [ ]:
# Configure parallelism for copy operations
parallelism=10

# Read the file list and perform parallel copy operations
df = spark.read.option("header", "false").csv(file_list_location).toDF("source", "dest")
df.repartition(parallelism).foreachPartition(copy_files_in_partition)

Verify destination bucket listing

In [ ]:
create_s3_tree(target_bucket_name, f'{db_name}/{table_name}')

#### C. Replicate table metadata in the secondary region

Replicate table metadata

In [ ]:
replicate_table_metadata(
    source_database=db_name,
    source_table=table_name,
    source_region=region1,
    target_region=region2,
    source_bucket=bucket_name,
    target_bucket=target_bucket_name
)

Verify table was properly replicated in the other region

In [ ]:
spark.sql(f"""
SELECT * 
FROM iceberg_catalog_region2.{db_name}.{table_name} ORDER BY id
""").show(truncate=False)

### Step2: Incremental Rewrite

We'll now simulate table write operations that create new snapshots by performing two MERGE statements:

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema1 = StructType([
    StructField("id", IntegerType(), False),
    StructField("col1", IntegerType(), True),
    StructField("col2", IntegerType(), True),
    StructField("col3", IntegerType(), True),
    StructField("col4", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("operation", StringType(), False)
])

# Create data for the first MERGE operation
data1 = [
    # New record to insert
    (5, 104, 204, 304, 404, 'italy', 'insert'),
    # Record to delete
    (4, None, None, None, None, 'france', 'delete')
]

# Create DataFrame for the first MERGE operation
merge_df1 = spark.createDataFrame(data1, schema1)
merge_df1.createOrReplaceTempView("merge_source1")

spark.sql(f"""
    MERGE INTO iceberg_catalog_region1.{db_name}.{table_name} t
    USING merge_source1 s
    ON t.id = s.id AND t.country = s.country
    WHEN MATCHED AND s.operation = 'update' THEN
        UPDATE SET t.col1 = s.col1, t.col2 = s.col2, t.col3 = s.col3, t.col4 = s.col4
    WHEN MATCHED AND s.operation = 'delete' THEN 
        DELETE
    WHEN NOT MATCHED AND s.operation = 'insert' THEN
        INSERT (id, col1, col2, col3, col4, country)
        VALUES (s.id, s.col1, s.col2, s.col3, s.col4, s.country)
""")

# Create data for the second MERGE operation
data2 = [
    # New record to insert
    (6, 106, 206, 306, 406, 'france', 'insert'),
    # Record to update
    (3, 999, 999, 999, 999, 'italy', 'update')
]

# Create DataFrame for the second MERGE operation
merge_df2 = spark.createDataFrame(data2, schema1)
merge_df2.createOrReplaceTempView("merge_source2")

spark.sql(f"""
    MERGE INTO iceberg_catalog_region1.{db_name}.{table_name} t
    USING merge_source2 s
    ON t.id = s.id AND t.country = s.country
    WHEN MATCHED AND s.operation = 'update' THEN
        UPDATE SET t.col1 = s.col1, t.col2 = s.col2, t.col3 = s.col3, t.col4 = s.col4
    WHEN MATCHED AND s.operation = 'delete' THEN 
        DELETE
    WHEN NOT MATCHED AND s.operation = 'insert' THEN
        INSERT (id, col1, col2, col3, col4, country)
        VALUES (s.id, s.col1, s.col2, s.col3, s.col4, s.country)
""")

Here the new table state

In [ ]:
spark.sql(f"""
SELECT * 
FROM iceberg_catalog_region1.{db_name}.{table_name} ORDER BY id
""").show(truncate=False)

Here the new S3 state

In [ ]:
create_s3_tree(bucket_name, f'{db_name}/{table_name}')

#### A. Invoke the rewrite_table_path procedure (Incremental rewrite)

Invoke the `rewrite_table_path` procedure to create Iceberg metadata files with `source_prefix` substituted by `target_prefix` - incrementally from the last run

The last execution of `rewrite_table_path` procedure returned the last rewritten metadata.json file name that was saved in the `latest_version` variable.

We can now trigger the Incremental rewrite using `latest_version` as `start_version`

In [ ]:
rewrite_table_path_incremental_df=spark.sql(f"""
CALL iceberg_catalog_region1.system.rewrite_table_path(
    table => '{db_name}.{table_name}', 
    source_prefix => '{s3_prefix_source}',
    target_prefix => '{s3_prefix_target}',
    start_version => '{latest_version}'
)
"""
)

rows = rewrite_table_path_incremental_df.collect()
file_list_location = rows[0]["file_list_location"]
latest_version = rows[0]["latest_version"]

In [ ]:
rewrite_table_path_incremental_df.show(truncate=False)

In [ ]:
Here the new S3 state

In [ ]:
create_s3_tree(bucket_name, f'{db_name}/{table_name}')

#### B. Copy table files in secondary region

Copy the new files in the destination bucket

In [ ]:
# Configure parallelism for copy operations
parallelism=10

# Read the file list and perform parallel copy operations
df = spark.read.option("header", "false").csv(file_list_location).toDF("source", "dest")
df.repartition(parallelism).foreachPartition(copy_files_in_partition)

Verify destination bucket listing

In [ ]:
create_s3_tree(target_bucket_name, f'{db_name}/{table_name}')

#### C. Replicate table metadata in the secondary region

In [ ]:
replicate_table_metadata(
    source_database=db_name,
    source_table=table_name,
    source_region=region1,
    target_region=region2,
    source_bucket=bucket_name,
    target_bucket=target_bucket_name
)

Verify table was properly replicated in the other region

In [ ]:
spark.sql(f"""
SELECT * 
FROM iceberg_catalog_region2.{db_name}.{table_name} ORDER BY id
""").show(truncate=False)